# Liquidity Models — Hypothesis Validation

**Thesis:** Pure OHLCV-based TA models plateau at composite 0.60-0.66. Liquidity and positioning data are orthogonal signal sources that measure *who* is doing what, not just *what* price did.

## Models Under Test
| # | Model | Data Source | Core Signal |
|---|-------|-------------|-------------|
| 1 | Taker Flow Imbalance (TFI) | Kline taker_buy fields (already fetched) | Aggressive buyer/seller dominance |
| 2 | Funding Rate Mean-Reversion (FRM) | `/fapi/v1/fundingRate` | Overleveraged crowd positioning |
| 3 | Open Interest Divergence (OID) | `/futures/data/openInterestHist` | OI vs price divergence |
| 4 | Liquidation Cascade (LC) | Taker buy volume spikes as proxy | Forced liquidation momentum |
| 5 | Volume Profile / VPOC | Constructed from klines | Price reversion to high-volume zones |

**Go/No-Go per model:** Forward return correlation |r| > 0.03, win rate > 53%, positive expectancy, orthogonal to existing models.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import time
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

from binance.um_futures import UMFutures
from libs.features.indicators.momentum.rsi import _compute_rsi_batch
from libs.features.indicators.volatility.atr import _compute_atr_batch
from libs.features.indicators.trend.ema import _compute_ema_batch

plt.style.use('dark_background')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

BASE_URL = 'https://fapi.binance.com'
client = UMFutures()
print('Setup complete')

---
## 0. Data Fetch — OHLCV with Taker Fields + Derivatives Data

In [ ]:
# ── OHLCV fetch (KEEP taker columns this time) ────────────────────
_ALL_COLS = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore',
]
_KEEP_COLS = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
              'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote']
_MAX_LIMIT = 1500

def fetch_ohlcv_full(
    symbol: str,
    timeframe: str,
    start_date: str = '2025-06-01',
    end_date: str | None = None,
) -> pd.DataFrame:
    """Fetch OHLCV + taker fields from Binance Futures.

    Args:
        symbol: Trading pair (e.g. 'BTCUSDT').
        timeframe: Kline interval (e.g. '1h', '4h').
        start_date: Start date string 'YYYY-MM-DD' (UTC).
        end_date: End date string 'YYYY-MM-DD' (UTC). Defaults to now.
    """
    since = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    if end_date:
        end = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    else:
        end = int(datetime.now(timezone.utc).timestamp() * 1000)
    frames = []
    cursor = since
    while cursor < end:
        lines = client.klines(symbol, timeframe, startTime=cursor, endTime=end, limit=_MAX_LIMIT)
        if not lines:
            break
        df = pd.DataFrame(lines, columns=_ALL_COLS)[_KEEP_COLS]
        for c in _KEEP_COLS:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        frames.append(df)
        last_ts = int(df['timestamp'].iloc[-1])
        if last_ts <= cursor:
            break
        cursor = last_ts + 1
        if len(lines) < _MAX_LIMIT:
            break
    if not frames:
        return pd.DataFrame(columns=_KEEP_COLS)
    result = pd.concat(frames, ignore_index=True)
    result = result.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)
    result['dt'] = pd.to_datetime(result['timestamp'], unit='ms', utc=True)
    result = result.set_index('dt')
    print(f"{symbol} {timeframe}: {len(result)} candles ({result.index[0].date()} to {result.index[-1].date()})")
    return result

# Fetch 1 year 1h data for BTC and ETH
btc = fetch_ohlcv_full('BTCUSDT', '1h', start_date='2025-06-01')
eth = fetch_ohlcv_full('ETHUSDT', '1h', start_date='2025-06-01')
print(f"\nTaker columns present: {['taker_buy_base', 'taker_buy_quote']}")
print(f"BTC taker_buy_base sample: {btc['taker_buy_base'].iloc[-3:].values}")

In [ ]:
# ── Fetch Funding Rate history ────────────────────────────────────
def fetch_funding_rate(
    symbol: str,
    start_date: str = '2025-06-01',
    end_date: str | None = None,
) -> pd.DataFrame:
    """Fetch funding rate history (every 8h) with pagination."""
    since = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    end = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000) if end_date else int(datetime.now(timezone.utc).timestamp() * 1000)
    frames = []
    cursor = since
    while cursor < end:
        resp = requests.get(f'{BASE_URL}/fapi/v1/fundingRate', params={
            'symbol': symbol, 'startTime': cursor, 'endTime': end, 'limit': 1000
        })
        data = resp.json()
        if not data:
            break
        df = pd.DataFrame(data)
        frames.append(df)
        cursor = int(df['fundingTime'].iloc[-1]) + 1
        if len(data) < 1000:
            break
        time.sleep(0.2)
    if not frames:
        return pd.DataFrame()
    result = pd.concat(frames, ignore_index=True)
    result['fundingRate'] = result['fundingRate'].astype(float)
    result['fundingTime'] = pd.to_datetime(result['fundingTime'].astype(int), unit='ms', utc=True)
    if 'markPrice' in result.columns:
        result['markPrice'] = pd.to_numeric(result['markPrice'], errors='coerce')
    result = result.set_index('fundingTime').sort_index()
    print(f"{symbol} funding: {len(result)} records ({result.index[0].date()} to {result.index[-1].date()})")
    return result

funding_btc = fetch_funding_rate('BTCUSDT', start_date='2025-06-01')
funding_eth = fetch_funding_rate('ETHUSDT', start_date='2025-06-01')

In [ ]:
# ── Fetch Open Interest history ──────────────────────────────────
def fetch_oi_hist(
    symbol: str,
    period: str = '1h',
    start_date: str | None = None,
    end_date: str | None = None,
    days: int = 30,
) -> pd.DataFrame:
    """Fetch OI statistics. API only provides last 30 days.

    If start_date is given, uses it; otherwise falls back to `days` ago.
    """
    now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
    if start_date:
        since = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    else:
        since = now_ms - days * 86_400_000
    end = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000) if end_date else now_ms
    frames = []
    cursor = since
    while cursor < end:
        resp = requests.get(f'{BASE_URL}/futures/data/openInterestHist', params={
            'symbol': symbol, 'period': period, 'startTime': cursor,
            'endTime': end, 'limit': 500
        })
        data = resp.json()
        if not data or not isinstance(data, list):
            break
        df = pd.DataFrame(data)
        frames.append(df)
        cursor = int(df['timestamp'].iloc[-1]) + 1
        if len(data) < 500:
            break
        time.sleep(0.2)
    if not frames:
        return pd.DataFrame()
    result = pd.concat(frames, ignore_index=True)
    result['sumOpenInterest'] = result['sumOpenInterest'].astype(float)
    result['sumOpenInterestValue'] = result['sumOpenInterestValue'].astype(float)
    result['dt'] = pd.to_datetime(result['timestamp'].astype(int), unit='ms', utc=True)
    result = result.set_index('dt').sort_index()
    print(f"{symbol} OI: {len(result)} records ({result.index[0].date()} to {result.index[-1].date()})")
    return result

# OI API only serves last ~30 days regardless of start_date
oi_btc = fetch_oi_hist('BTCUSDT', '1h')
oi_eth = fetch_oi_hist('ETHUSDT', '1h')

In [ ]:
# ── Fetch Long/Short Ratio ───────────────────────────────────────
def fetch_ls_ratio(
    symbol: str,
    period: str = '1h',
    start_date: str | None = None,
    end_date: str | None = None,
    days: int = 30,
) -> pd.DataFrame:
    """Fetch global long/short account ratio. Last 30 days only.

    If start_date is given, uses it; otherwise falls back to `days` ago.
    """
    now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
    if start_date:
        since = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    else:
        since = now_ms - days * 86_400_000
    end = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000) if end_date else now_ms
    frames = []
    cursor = since
    while cursor < end:
        resp = requests.get(f'{BASE_URL}/futures/data/globalLongShortAccountRatio', params={
            'symbol': symbol, 'period': period, 'startTime': cursor,
            'endTime': end, 'limit': 500
        })
        data = resp.json()
        if not data or not isinstance(data, list):
            break
        df = pd.DataFrame(data)
        frames.append(df)
        cursor = int(df['timestamp'].iloc[-1]) + 1
        if len(data) < 500:
            break
        time.sleep(0.2)
    if not frames:
        return pd.DataFrame()
    result = pd.concat(frames, ignore_index=True)
    for col in ['longShortRatio', 'longAccount', 'shortAccount']:
        result[col] = result[col].astype(float)
    result['dt'] = pd.to_datetime(result['timestamp'].astype(int), unit='ms', utc=True)
    result = result.set_index('dt').sort_index()
    print(f"{symbol} L/S: {len(result)} records ({result.index[0].date()} to {result.index[-1].date()})")
    return result

# L/S API only serves last ~30 days regardless of start_date
ls_btc = fetch_ls_ratio('BTCUSDT', '1h')
ls_eth = fetch_ls_ratio('ETHUSDT', '1h')

In [ ]:
# ── Helper: forward returns ──────────────────────────────────────
def add_fwd_returns(df: pd.DataFrame, horizons: list = [1, 4, 8, 12, 24]) -> pd.DataFrame:
    out = df.copy()
    for h in horizons:
        out[f'fwd_{h}'] = out['close'].pct_change(h).shift(-h)
    return out

# Compute RSI and ATR for context/filtering
for df in [btc, eth]:
    df['RSI'] = _compute_rsi_batch(df['close'].values, 14)
    df['ATR'] = _compute_atr_batch(df['high'].values, df['low'].values, df['close'].values, 14)

btc = add_fwd_returns(btc)
eth = add_fwd_returns(eth)
print(f"Forward returns + RSI/ATR computed. BTC shape: {btc.shape}")

---
# MODEL 1: Taker Flow Imbalance (TFI)

**Thesis:** When aggressive buyers dominate (taker_buy_base / volume > 0.55-0.60), directional conviction precedes price moves. Unlike raw volume, TFI reveals *who* is aggressing across the spread.

In [ ]:
# ── TFI computation ──────────────────────────────────────────────
def compute_tfi(df: pd.DataFrame, smooth: int = 5) -> pd.DataFrame:
    out = df.copy()
    # Raw taker buy ratio
    out['tfi_raw'] = out['taker_buy_base'] / out['volume'].replace(0, np.nan)
    # Smoothed (EMA-like rolling)
    out['tfi'] = out['tfi_raw'].ewm(span=smooth, adjust=False).mean()
    # Taker sell ratio
    out['taker_sell_base'] = out['volume'] - out['taker_buy_base']
    # Net taker flow (buy - sell, normalized)
    out['tfi_net'] = (out['taker_buy_base'] - out['taker_sell_base']) / out['volume'].replace(0, np.nan)
    out['tfi_net_smooth'] = out['tfi_net'].ewm(span=smooth, adjust=False).mean()
    # TFI z-score (how extreme is current vs recent history)
    roll_mean = out['tfi'].rolling(100, min_periods=20).mean()
    roll_std = out['tfi'].rolling(100, min_periods=20).std()
    out['tfi_zscore'] = (out['tfi'] - roll_mean) / roll_std.replace(0, np.nan)
    return out

btc = compute_tfi(btc)
eth = compute_tfi(eth)

print('TFI Distribution (BTC):')
print(btc['tfi'].describe())
print(f"\nTFI range: [{btc['tfi'].min():.4f}, {btc['tfi'].max():.4f}]")
print(f"Mean: {btc['tfi'].mean():.4f} (0.50 = neutral)")

In [ ]:
# ── TFI vs Forward Returns — Correlation & Conditional Returns ──
print("=" * 70)
print("MODEL 1: TAKER FLOW IMBALANCE (TFI) — Predictive Power")
print("=" * 70)

# 1a. Correlation
print("\n▸ TFI vs Forward Returns (Pearson r):")
for label, data in [('BTC', btc), ('ETH', eth)]:
    print(f"  {label}:")
    for tfi_col in ['tfi', 'tfi_net_smooth', 'tfi_zscore']:
        for fwd_col in ['fwd_4', 'fwd_8', 'fwd_12', 'fwd_24']:
            corr = data[[tfi_col, fwd_col]].dropna().corr().iloc[0, 1]
            marker = ' ◀' if abs(corr) > 0.03 else ''
            print(f"    {tfi_col:18s} vs {fwd_col}: r={corr:+.4f}{marker}")

# 1b. Conditional returns by TFI quintile
print("\n▸ Forward 12h Return by TFI Quintile (BTC):")
btc['tfi_quintile'] = pd.qcut(btc['tfi'].dropna(), 5, labels=['Q1_sell', 'Q2', 'Q3_neutral', 'Q4', 'Q5_buy'])
quintile_stats = btc.groupby('tfi_quintile', observed=True)['fwd_12'].agg(['mean', 'std', 'count'])
quintile_stats['mean_pct'] = quintile_stats['mean'] * 100
quintile_stats['wr'] = btc.groupby('tfi_quintile', observed=True)['fwd_12'].apply(lambda x: (x > 0).mean()) * 100
print(quintile_stats[['mean_pct', 'wr', 'count']].to_string())

In [ ]:
# ── TFI Signal Backtest ──────────────────────────────────────────
def backtest_tfi(
    df: pd.DataFrame,
    tfi_long_thresh: float = 0.55,
    tfi_short_thresh: float = 0.45,
    use_zscore: bool = False,
    zscore_long: float = 1.0,
    zscore_short: float = -1.0,
    atr_tp: float = 2.0,
    atr_sl: float = 1.5,
    cooldown: int = 6,
) -> pd.DataFrame:
    closes = df['close'].values
    highs = df['high'].values
    lows = df['low'].values
    atr = df['ATR'].values
    tfi = df['tfi_zscore'].values if use_zscore else df['tfi'].values
    rsi = df['RSI'].values
    t_long = zscore_long if use_zscore else tfi_long_thresh
    t_short = zscore_short if use_zscore else tfi_short_thresh

    trades = []
    pos = 0
    entry_px = tp_px = sl_px = 0.0
    entry_bar = 0
    last_exit = -cooldown

    for i in range(50, len(df)):
        if np.isnan(atr[i]) or np.isnan(tfi[i]) or np.isnan(rsi[i]):
            continue
        # Exits
        if pos == 1:
            if lows[i] <= sl_px:
                trades.append({'bar': entry_bar, 'exit': i, 'dir': 'L', 'pnl': (sl_px - entry_px)/entry_px*100, 'r': 'SL'})
                pos = 0; last_exit = i
            elif highs[i] >= tp_px:
                trades.append({'bar': entry_bar, 'exit': i, 'dir': 'L', 'pnl': (tp_px - entry_px)/entry_px*100, 'r': 'TP'})
                pos = 0; last_exit = i
        elif pos == -1:
            if highs[i] >= sl_px:
                trades.append({'bar': entry_bar, 'exit': i, 'dir': 'S', 'pnl': (entry_px - sl_px)/entry_px*100, 'r': 'SL'})
                pos = 0; last_exit = i
            elif lows[i] <= tp_px:
                trades.append({'bar': entry_bar, 'exit': i, 'dir': 'S', 'pnl': (entry_px - tp_px)/entry_px*100, 'r': 'TP'})
                pos = 0; last_exit = i
        # Entries
        if pos == 0 and (i - last_exit) >= cooldown:
            if tfi[i] > t_long and rsi[i] < 70:
                pos = 1; entry_px = closes[i]
                tp_px = entry_px + atr[i]*atr_tp; sl_px = entry_px - atr[i]*atr_sl; entry_bar = i
            elif tfi[i] < t_short and rsi[i] > 30:
                pos = -1; entry_px = closes[i]
                tp_px = entry_px - atr[i]*atr_tp; sl_px = entry_px + atr[i]*atr_sl; entry_bar = i
    return pd.DataFrame(trades)

def print_bt(label, trades):
    if trades.empty:
        print(f"  {label}: No trades"); return
    n = len(trades)
    wr = (trades['pnl'] > 0).mean() * 100
    avg = trades['pnl'].mean()
    tot = trades['pnl'].sum()
    aw = trades.loc[trades['pnl']>0, 'pnl'].mean() if (trades['pnl']>0).any() else 0
    al = trades.loc[trades['pnl']<=0, 'pnl'].mean() if (trades['pnl']<=0).any() else 0
    pf = abs(aw/al) if al != 0 else float('inf')
    print(f"  {label}: {n} trades | WR={wr:.1f}% | Avg={avg:+.3f}% | Total={tot:+.2f}% | PF={pf:.2f}")

# Threshold-based
print("\n▸ TFI Backtest (threshold-based):")
for thresh in [0.52, 0.55, 0.58, 0.60]:
    t = backtest_tfi(btc, tfi_long_thresh=thresh, tfi_short_thresh=1-thresh)
    print_bt(f'BTC tfi>{thresh:.2f}', t)

# Z-score based
print("\n▸ TFI Backtest (z-score based):")
for z in [0.5, 1.0, 1.5, 2.0]:
    t = backtest_tfi(btc, use_zscore=True, zscore_long=z, zscore_short=-z)
    print_bt(f'BTC z>{z:.1f}', t)

# Best config on ETH
print("\n▸ Best TFI config on ETH (out-of-sample asset):")
t_eth = backtest_tfi(eth, use_zscore=True, zscore_long=1.0, zscore_short=-1.0)
print_bt('ETH z>1.0', t_eth)

In [ ]:
# ── TFI Equity Curve ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (label, df_data) in zip(axes, [('BTC', btc), ('ETH', eth)]):
    t = backtest_tfi(df_data, use_zscore=True, zscore_long=1.0, zscore_short=-1.0)
    if t.empty:
        ax.set_title(f'{label} TFI: No trades'); continue
    ax.plot(t['pnl'].cumsum().values, linewidth=1.5)
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.set_title(f'{label} TFI z>1.0 ({len(t)} trades, WR={(t["pnl"]>0).mean()*100:.0f}%)')
    ax.set_xlabel('Trade #'); ax.set_ylabel('Cum PnL (%)')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
# MODEL 2: Funding Rate Mean-Reversion (FRM)

**Thesis:** Extreme funding rates (>0.05% or <-0.03%) indicate overleveraged positioning. The crowd pays to hold → they're wrong → price mean-reverts within 8-24h.

In [ ]:
# ── Merge funding with OHLCV ─────────────────────────────────────
def merge_funding(ohlcv: pd.DataFrame, funding: pd.DataFrame) -> pd.DataFrame:
    """Forward-fill funding rate into hourly OHLCV."""
    out = ohlcv.copy()
    # Reindex funding to hourly, forward-fill (funding is every 8h)
    fr = funding[['fundingRate']].copy()
    out = out.join(fr, how='left')
    out['fundingRate'] = out['fundingRate'].ffill()
    # Cumulative funding (rolling 24h = 3 funding periods)
    out['funding_24h'] = out['fundingRate'].rolling(24, min_periods=1).sum() / 3  # approx 3 periods in 24h
    # Funding z-score
    fr_mean = out['fundingRate'].rolling(24*7, min_periods=24).mean()  # 1 week lookback
    fr_std = out['fundingRate'].rolling(24*7, min_periods=24).std()
    out['funding_zscore'] = (out['fundingRate'] - fr_mean) / fr_std.replace(0, np.nan)
    return out

btc = merge_funding(btc, funding_btc)
eth = merge_funding(eth, funding_eth)

print("Funding Rate Distribution (BTC):")
print(btc['fundingRate'].dropna().describe())
print(f"\nExtreme positive (>0.0005): {(btc['fundingRate'] > 0.0005).sum()} bars")
print(f"Extreme negative (<-0.0003): {(btc['fundingRate'] < -0.0003).sum()} bars")

In [ ]:
# ── Funding Rate vs Forward Returns ──────────────────────────────
print("=" * 70)
print("MODEL 2: FUNDING RATE MEAN-REVERSION — Predictive Power")
print("=" * 70)

# Correlation
print("\n▸ Funding Rate vs Forward Returns (Pearson r):")
for label, data in [('BTC', btc), ('ETH', eth)]:
    print(f"  {label}:")
    for fr_col in ['fundingRate', 'funding_zscore']:
        for fwd_col in ['fwd_8', 'fwd_12', 'fwd_24']:
            corr = data[[fr_col, fwd_col]].dropna().corr().iloc[0, 1]
            marker = ' ◀' if abs(corr) > 0.03 else ''
            print(f"    {fr_col:18s} vs {fwd_col}: r={corr:+.4f}{marker}")

# Conditional: extreme funding → forward returns
print("\n▸ Extreme Funding → Forward Returns (BTC):")
thresholds = [
    ('funding > +0.05%', btc['fundingRate'] > 0.0005, 'Short signal'),
    ('funding > +0.03%', btc['fundingRate'] > 0.0003, 'Short signal'),
    ('funding < -0.03%', btc['fundingRate'] < -0.0003, 'Long signal'),
    ('funding < -0.01%', btc['fundingRate'] < -0.0001, 'Long signal'),
]
for label, mask, direction in thresholds:
    subset = btc.loc[mask]
    if len(subset) < 5:
        print(f"  {label} ({direction}): n={len(subset)} — too few"); continue
    for h in [8, 12, 24]:
        fwd = subset[f'fwd_{h}'].dropna()
        is_short = 'Short' in direction
        wr = ((fwd < 0).mean() if is_short else (fwd > 0).mean()) * 100
        print(f"  {label} ({direction}) → {h}h: mean={fwd.mean()*100:+.4f}% WR={wr:.1f}% n={len(fwd)}")

In [ ]:
# ── Funding Rate Backtest ────────────────────────────────────────
def backtest_funding(
    df: pd.DataFrame,
    short_thresh: float = 0.0005,  # funding > this → short (longs overleveraged)
    long_thresh: float = -0.0003,  # funding < this → long (shorts overleveraged)
    atr_tp: float = 2.0,
    atr_sl: float = 1.5,
    cooldown: int = 8,
) -> pd.DataFrame:
    closes = df['close'].values
    highs = df['high'].values
    lows = df['low'].values
    atr = df['ATR'].values
    fr = df['fundingRate'].values
    rsi = df['RSI'].values

    trades = []
    pos = 0; entry_px = tp_px = sl_px = 0.0; entry_bar = 0; last_exit = -cooldown

    for i in range(50, len(df)):
        if np.isnan(atr[i]) or np.isnan(rsi[i]) or np.isnan(fr[i]):
            continue
        if pos == 1:
            if lows[i] <= sl_px:
                trades.append({'bar': entry_bar, 'exit': i, 'dir': 'L', 'pnl': (sl_px-entry_px)/entry_px*100, 'r': 'SL'})
                pos = 0; last_exit = i
            elif highs[i] >= tp_px:
                trades.append({'bar': entry_bar, 'exit': i, 'dir': 'L', 'pnl': (tp_px-entry_px)/entry_px*100, 'r': 'TP'})
                pos = 0; last_exit = i
        elif pos == -1:
            if highs[i] >= sl_px:
                trades.append({'bar': entry_bar, 'exit': i, 'dir': 'S', 'pnl': (entry_px-sl_px)/entry_px*100, 'r': 'SL'})
                pos = 0; last_exit = i
            elif lows[i] <= tp_px:
                trades.append({'bar': entry_bar, 'exit': i, 'dir': 'S', 'pnl': (entry_px-tp_px)/entry_px*100, 'r': 'TP'})
                pos = 0; last_exit = i
        if pos == 0 and (i - last_exit) >= cooldown:
            # Counter-trend: high funding → short, low funding → long
            if fr[i] > short_thresh and rsi[i] > 50:  # funding high + RSI confirms overbought
                pos = -1; entry_px = closes[i]
                tp_px = entry_px - atr[i]*atr_tp; sl_px = entry_px + atr[i]*atr_sl; entry_bar = i
            elif fr[i] < long_thresh and rsi[i] < 50:  # funding low + RSI confirms oversold
                pos = 1; entry_px = closes[i]
                tp_px = entry_px + atr[i]*atr_tp; sl_px = entry_px - atr[i]*atr_sl; entry_bar = i
    return pd.DataFrame(trades)

print("\n▸ Funding Rate Backtest:")
configs = [
    (0.0005, -0.0003, 'strict'),
    (0.0003, -0.0002, 'moderate'),
    (0.0002, -0.0001, 'loose'),
]
for s_t, l_t, name in configs:
    for label, data in [('BTC', btc), ('ETH', eth)]:
        t = backtest_funding(data, short_thresh=s_t, long_thresh=l_t)
        print_bt(f'{label} {name} (>{s_t*100:.2f}%/<{l_t*100:.2f}%)', t)

In [ ]:
# ── Funding Rate Equity Curve ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (label, data) in zip(axes, [('BTC', btc), ('ETH', eth)]):
    t = backtest_funding(data, short_thresh=0.0003, long_thresh=-0.0002)
    if t.empty:
        ax.set_title(f'{label} FRM: No trades'); continue
    ax.plot(t['pnl'].cumsum().values, linewidth=1.5)
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.set_title(f'{label} Funding MR ({len(t)} trades, WR={(t["pnl"]>0).mean()*100:.0f}%)')
    ax.set_xlabel('Trade #'); ax.set_ylabel('Cum PnL (%)')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
# MODEL 3: Open Interest Divergence (OID)

**Thesis:** When price rises but OI falls → short covering (weak rally, fade it). When price rises and OI rises → new conviction longs (ride it). OI/price divergence predicts reversals.

*Note: OI data limited to last 30 days.*

In [ ]:
# ── Merge OI with OHLCV ──────────────────────────────────────────
def merge_oi(ohlcv: pd.DataFrame, oi: pd.DataFrame) -> pd.DataFrame:
    """Join OI data to hourly OHLCV."""
    out = ohlcv.copy()
    if oi.empty:
        out['oi'] = np.nan; out['oi_change'] = np.nan; out['oi_pct_change'] = np.nan
        out['px_change'] = np.nan; out['oi_px_div'] = np.nan; out['oi_div_score'] = np.nan
        return out
    oi_col = oi[['sumOpenInterest']].rename(columns={'sumOpenInterest': 'oi'})
    out = out.join(oi_col, how='left')
    out['oi'] = out['oi'].interpolate(method='time')
    out['oi_change'] = out['oi'].diff()
    out['oi_pct_change'] = out['oi'].pct_change()
    # Price change for divergence detection
    out['px_change'] = out['close'].pct_change()
    # Divergence: price up + OI down, or price down + OI up
    out['oi_px_div'] = np.where(
        (out['px_change'] > 0) & (out['oi_change'] < 0), -1,  # bearish divergence (short covering)
        np.where(
            (out['px_change'] < 0) & (out['oi_change'] > 0), 1,  # bearish conviction (new shorts)
            np.where(
                (out['px_change'] > 0) & (out['oi_change'] > 0), 2,  # bullish conviction
                0  # no signal
            )
        )
    )
    # Rolling divergence score
    out['oi_div_score'] = out['oi_px_div'].rolling(4, min_periods=1).mean()
    return out

# Trim OHLCV to OI period for comparison
btc_oi = merge_oi(btc, oi_btc)
eth_oi = merge_oi(eth, oi_eth)

# Only keep rows where OI data exists
btc_oi_valid = btc_oi.dropna(subset=['oi'])
eth_oi_valid = eth_oi.dropna(subset=['oi'])
print(f"BTC OI-valid rows: {len(btc_oi_valid)}, ETH: {len(eth_oi_valid)}")
if not btc_oi_valid.empty:
    print(f"OI divergence distribution (BTC):")
    print(btc_oi_valid['oi_px_div'].value_counts().to_dict())
else:
    print("No OI data available — API may have returned empty (30-day limit)")

In [ ]:
# ── OI Divergence vs Forward Returns ─────────────────────────────
print("=" * 70)
print("MODEL 3: OPEN INTEREST DIVERGENCE — Predictive Power")
print("=" * 70)

for label, data in [('BTC', btc_oi_valid), ('ETH', eth_oi_valid)]:
    if len(data) < 50:
        print(f"  {label}: Insufficient OI data ({len(data)} rows)"); continue
    print(f"\n  {label} (n={len(data)} bars with OI data):")
    
    states = {
        'Short covering (px↑ OI↓)': data[data['oi_px_div'] == -1],
        'New shorts (px↓ OI↑)': data[data['oi_px_div'] == 1],
        'Bullish conviction (px↑ OI↑)': data[data['oi_px_div'] == 2],
        'Neutral': data[data['oi_px_div'] == 0],
    }
    
    for state_name, subset in states.items():
        if len(subset) < 5:
            continue
        for h in [4, 8, 12]:
            fwd = subset[f'fwd_{h}'].dropna()
            if len(fwd) < 5:
                continue
            wr_up = (fwd > 0).mean() * 100
            print(f"    {state_name:35s} → {h}h: mean={fwd.mean()*100:+.4f}% WR_up={wr_up:.1f}% n={len(fwd)}")

# Correlation
print("\n▸ OI change vs Forward Returns:")
for label, data in [('BTC', btc_oi_valid), ('ETH', eth_oi_valid)]:
    if len(data) < 50:
        continue
    for fwd_col in ['fwd_4', 'fwd_8', 'fwd_12']:
        corr = data[['oi_pct_change', fwd_col]].dropna().corr().iloc[0, 1]
        marker = ' ◀' if abs(corr) > 0.03 else ''
        print(f"  {label} oi_pct_change vs {fwd_col}: r={corr:+.4f}{marker}")

---
# MODEL 4: Liquidation Cascade Proxy

**Thesis:** Large liquidation events create forced buying/selling. We proxy this via sudden taker volume spikes + extreme price moves — a volume shock model.

*Note: No direct liquidation API for historical data in REST. Using taker volume spike as proxy.*

In [ ]:
# ── Liquidation Proxy: Volume Spike + Price Shock ────────────────
def compute_liq_proxy(df: pd.DataFrame, vol_lookback: int = 50, spike_mult: float = 3.0) -> pd.DataFrame:
    out = df.copy()
    # Volume z-score
    vol_mean = out['volume'].rolling(vol_lookback, min_periods=10).mean()
    vol_std = out['volume'].rolling(vol_lookback, min_periods=10).std()
    out['vol_zscore'] = (out['volume'] - vol_mean) / vol_std.replace(0, np.nan)
    # Price shock (abs return z-score)
    ret = out['close'].pct_change().abs()
    ret_mean = ret.rolling(vol_lookback, min_periods=10).mean()
    ret_std = ret.rolling(vol_lookback, min_periods=10).std()
    out['ret_zscore'] = (ret - ret_mean) / ret_std.replace(0, np.nan)
    # Combined liquidation proxy score
    out['liq_score'] = out['vol_zscore'] * out['ret_zscore']
    # Direction of the shock
    out['shock_dir'] = np.sign(out['close'].pct_change())
    # Spike events
    out['vol_spike'] = (out['vol_zscore'] > spike_mult).astype(int)
    out['liq_event'] = ((out['vol_zscore'] > spike_mult) & (out['ret_zscore'] > 2.0)).astype(int)
    return out

btc = compute_liq_proxy(btc)
eth = compute_liq_proxy(eth)

print("=" * 70)
print("MODEL 4: LIQUIDATION CASCADE PROXY — Signal Detection")
print("=" * 70)
for label, data in [('BTC', btc), ('ETH', eth)]:
    n_spikes = data['vol_spike'].sum()
    n_liq = data['liq_event'].sum()
    months = len(data) / (24 * 30)
    print(f"  {label}: {n_spikes} vol spikes ({n_spikes/months:.1f}/mo), {n_liq} liq events ({n_liq/months:.1f}/mo)")

# Forward returns after liquidation events
print("\n▸ Forward Returns After Liquidation Events:")
for label, data in [('BTC', btc), ('ETH', eth)]:
    liq_up = data[(data['liq_event'] == 1) & (data['shock_dir'] > 0)]
    liq_dn = data[(data['liq_event'] == 1) & (data['shock_dir'] < 0)]
    print(f"  {label} — Bullish liquidation events: {len(liq_up)}, Bearish: {len(liq_dn)}")
    for h in [1, 4, 8, 12]:
        if len(liq_up) >= 3:
            fwd = liq_up[f'fwd_{h}'].dropna()
            print(f"    Bull liq → {h}h: mean={fwd.mean()*100:+.4f}% WR_up={(fwd>0).mean()*100:.1f}% n={len(fwd)}")
        if len(liq_dn) >= 3:
            fwd = liq_dn[f'fwd_{h}'].dropna()
            print(f"    Bear liq → {h}h: mean={fwd.mean()*100:+.4f}% WR_dn={(fwd<0).mean()*100:.1f}% n={len(fwd)}")

---
# MODEL 5: Volume Profile / VPOC

**Thesis:** Price reverts to the Volume Point of Control (highest-volume price level over a rolling window). Deviation from VPOC is mean-reverting.

In [ ]:
# ── Volume Profile & VPOC ────────────────────────────────────────
def compute_vpoc(df: pd.DataFrame, lookback: int = 100, n_bins: int = 50) -> pd.DataFrame:
    """Rolling VPOC: price level with highest volume over lookback bars."""
    out = df.copy()
    vpoc = np.full(len(out), np.nan)
    closes = out['close'].values
    volumes = out['volume'].values
    highs = out['high'].values
    lows = out['low'].values

    for i in range(lookback, len(out)):
        window_low = lows[i-lookback:i].min()
        window_high = highs[i-lookback:i].max()
        if window_high == window_low:
            vpoc[i] = window_low
            continue
        # Build volume profile
        bins = np.linspace(window_low, window_high, n_bins + 1)
        vol_profile = np.zeros(n_bins)
        for j in range(i-lookback, i):
            # Distribute bar volume across price range
            bar_low_idx = max(0, int((lows[j] - window_low) / (window_high - window_low) * n_bins))
            bar_high_idx = min(n_bins - 1, int((highs[j] - window_low) / (window_high - window_low) * n_bins))
            if bar_high_idx >= bar_low_idx:
                spread = bar_high_idx - bar_low_idx + 1
                vol_profile[bar_low_idx:bar_high_idx+1] += volumes[j] / spread
        # VPOC = mid-price of highest volume bin
        max_bin = np.argmax(vol_profile)
        vpoc[i] = (bins[max_bin] + bins[max_bin + 1]) / 2

    out['vpoc'] = vpoc
    out['vpoc_dev'] = (out['close'] - out['vpoc']) / out['vpoc'] * 100  # % deviation
    out['vpoc_dev_atr'] = (out['close'] - out['vpoc']) / out['ATR'].replace(0, np.nan)  # ATR-normalized
    return out

btc = compute_vpoc(btc, lookback=100)
eth = compute_vpoc(eth, lookback=100)

print("VPOC Deviation Distribution (BTC):")
print(btc['vpoc_dev'].dropna().describe())

In [ ]:
# ── VPOC vs Forward Returns ──────────────────────────────────────
print("=" * 70)
print("MODEL 5: VOLUME PROFILE / VPOC — Predictive Power")
print("=" * 70)

# Correlation: VPOC deviation vs forward returns
print("\n▸ VPOC Deviation vs Forward Returns:")
for label, data in [('BTC', btc), ('ETH', eth)]:
    for fwd_col in ['fwd_4', 'fwd_8', 'fwd_12', 'fwd_24']:
        corr = data[['vpoc_dev', fwd_col]].dropna().corr().iloc[0, 1]
        marker = ' ◀' if abs(corr) > 0.03 else ''
        print(f"  {label} vpoc_dev vs {fwd_col}: r={corr:+.4f}{marker}")

# Conditional: extreme VPOC deviation → forward returns
print("\n▸ Extreme VPOC Deviation → Forward Returns (BTC):")
for thresh in [1.0, 1.5, 2.0, 2.5]:
    above = btc[btc['vpoc_dev_atr'] > thresh]
    below = btc[btc['vpoc_dev_atr'] < -thresh]
    for h in [8, 12, 24]:
        if len(above) >= 5:
            fwd = above[f'fwd_{h}'].dropna()
            print(f"  Above VPOC >{thresh} ATR → {h}h: mean={fwd.mean()*100:+.4f}% (expect negative if MR) n={len(fwd)}")
        if len(below) >= 5:
            fwd = below[f'fwd_{h}'].dropna()
            print(f"  Below VPOC <-{thresh} ATR → {h}h: mean={fwd.mean()*100:+.4f}% (expect positive if MR) n={len(fwd)}")

---
# 6. Cross-Model Correlation & Orthogonality

In [ ]:
# ── Signal orthogonality check ───────────────────────────────────
print("=" * 70)
print("CROSS-MODEL SIGNAL CORRELATION")
print("=" * 70)

# Collect all signal columns into one frame
signals = pd.DataFrame(index=btc.index)
signals['tfi'] = btc['tfi']
signals['tfi_zscore'] = btc['tfi_zscore']
signals['funding'] = btc['fundingRate']
signals['funding_z'] = btc['funding_zscore']
signals['vol_zscore'] = btc['vol_zscore']
signals['liq_score'] = btc['liq_score']
signals['vpoc_dev'] = btc['vpoc_dev']
signals['RSI'] = btc['RSI']

corr_matrix = signals.dropna().corr()
print("\n▸ Pairwise Pearson Correlation:")
print(corr_matrix.to_string(float_format='{:+.3f}'.format))

# Highlight high correlations
print("\n▸ High Correlations (|r| > 0.3):")
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.3:
            print(f"  {corr_matrix.index[i]} vs {corr_matrix.columns[j]}: r={r:+.3f} ⚠️")

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
ax.set_yticklabels(corr_matrix.columns)
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f'{corr_matrix.iloc[i,j]:+.2f}', ha='center', va='center',
                color='white' if abs(corr_matrix.iloc[i,j]) > 0.5 else 'black', fontsize=9)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Liquidity Signal Correlation Matrix (BTC/1h)')
plt.tight_layout(); plt.show()

---
# 7. Composite Summary & Go/No-Go

In [ ]:
# ── Final summary ────────────────────────────────────────────────
print("=" * 70)
print("LIQUIDITY MODELS — COMPOSITE SUMMARY")
print("=" * 70)

# TFI
t_tfi = backtest_tfi(btc, use_zscore=True, zscore_long=1.0, zscore_short=-1.0)
t_tfi_eth = backtest_tfi(eth, use_zscore=True, zscore_long=1.0, zscore_short=-1.0)

# Funding
t_frm = backtest_funding(btc, short_thresh=0.0003, long_thresh=-0.0002)
t_frm_eth = backtest_funding(eth, short_thresh=0.0003, long_thresh=-0.0002)

print("\n▸ Model Performance Summary:")
print(f"{'Model':<30} {'Asset':<6} {'Trades':>7} {'WR':>7} {'Avg PnL':>10} {'Total':>10} {'PF':>6}")
print("-" * 80)

def row(name, asset, t):
    if t.empty:
        print(f"{name:<30} {asset:<6} {'0':>7} {'—':>7} {'—':>10} {'—':>10} {'—':>6}")
        return
    n = len(t)
    wr = (t['pnl']>0).mean()*100
    avg = t['pnl'].mean()
    tot = t['pnl'].sum()
    aw = t.loc[t['pnl']>0, 'pnl'].mean() if (t['pnl']>0).any() else 0
    al = t.loc[t['pnl']<=0, 'pnl'].mean() if (t['pnl']<=0).any() else 0
    pf = abs(aw/al) if al != 0 else 0
    print(f"{name:<30} {asset:<6} {n:>7} {wr:>6.1f}% {avg:>+9.3f}% {tot:>+9.2f}% {pf:>5.2f}")

row('TFI (z>1.0)', 'BTC', t_tfi)
row('TFI (z>1.0)', 'ETH', t_tfi_eth)
row('Funding MR (moderate)', 'BTC', t_frm)
row('Funding MR (moderate)', 'ETH', t_frm_eth)

# OI summary (limited data)
print(f"\n{'OI Divergence':<30} {'BTC':<6} {'30d data only — see conditional returns above':>50}")
print(f"{'Liq Cascade Proxy':<30} {'BTC':<6} {'Event-driven — see conditional returns above':>50}")
print(f"{'VPOC':<30} {'BTC':<6} {'Correlation-only — see MR overlap analysis above':>50}")

print("\n▸ Go/No-Go Decision Criteria:")
print("  ✓ Forward correlation |r| > 0.03")
print("  ✓ Win rate > 53%")
print("  ✓ Positive total PnL on BOTH BTC and ETH")
print("  ✓ Orthogonal to existing models (|r| < 0.3 with RSI)")
print("  ✓ Signal density > 5/month")